In [ ]:
# step 0: preprocessing

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
from pathlib import Path

input_file_path = Path("../../data/demo/dummy_raw_dataset.xlsx")
output_file_path = Path("../../data/demo/dummy_transformed_dataset.xlsx")

target_column = "maltreatment y/n (CTQ)"

cat_threshold = 5

extreme_skew_cols = [
    "Avg videos not found (from favorite)",
    "Avg favorited",
    "Avg shares",
    "Avg direct messages sent",
    "Avg direct messages received",
    "Avg posts (count & manual) (AC)",
    "Being Liked (AC)",
    "Avg comments",
    "Avg single contacts",
    "Substanz-missbrauch",
    "SES_SUM",
]

moderate_skew_cols = [
    "Avg number of videos per week",
    "Avg skipped per week",
    "Avg unique per week",
    "Avg num multiple",
    "Avg liked per week",
    "Avg searches",
    "Avg no blocked",
    "Avg chat partners",
    "NSFW Score",
    "NSFW Score over night",
    "Avg session length (seconds)",
]

proportion_cols = [
    "Percentage videos watched over night",
    "Percentage videos watched on day",
]

binary_bin_cols = {
    "Parental control": lambda x: 1 if x > 0 else 0,
    "TT posts (self-report)": lambda x: 1 if x > 0 else 0,
    "TT only friends": lambda x: 1 if x > 0 else 0,
    "TT sexual solicitations": lambda x: 1 if x > 0 else 0,
}

gender_col = "Gender (1=m,2=f)"
gender_mapping = {1: 1, 2: 0}
diverse_label = "2"

new_categorical_cols = [
    "emoabuse_cat",
    "physabuse_cat",
    "sexabuse_cat",
    "emoneg_cat",
    "physneg_cat",
]


def compute_descriptive_stats(df, name=""):
    """compute descriptive statistics for numeric columns"""
    results = []
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    for col in numeric_cols:
        data = df[col].dropna()
        n = len(data)
        missing = df[col].isna().sum()

        if n == 0:
            continue

        mean_val = data.mean()
        median_val = data.median()
        std_dev = data.std()
        min_val = data.min()
        max_val = data.max()

        skewness = stats.skew(data)
        kurt = stats.kurtosis(data)

        se_skew = np.sqrt(6 / n) if n > 1 else np.nan
        se_kurt = np.sqrt(24 / n) if n > 1 else np.nan

        w_stat, p_val = np.nan, np.nan
        if 3 <= n <= 5000:
            w_stat, p_val = stats.shapiro(data)

        unique_count = df[col].nunique()

        col_type = (
            "Categorical"
            if unique_count <= cat_threshold
            else "Long-tailed (Continuous)"
        )

        results.append(
            {
                "Variable": col,
                "N": n,
                "Missing": missing,
                "Mean": round(mean_val, 4),
                "Median": round(median_val, 4),
                "Std Dev": round(std_dev, 4),
                "Min": round(min_val, 4),
                "Max": round(max_val, 4),
                "Skewness": round(skewness, 4),
                "Std. Error Skewness": round(se_skew, 4),
                "Kurtosis": round(kurt, 4),
                "Std. Error Kurtosis": round(se_kurt, 4),
                "Shapiro-Wilk W": round(w_stat, 4) if not np.isnan(w_stat) else "N/A",
                "Shapiro-Wilk p": round(p_val, 4) if not np.isnan(p_val) else "N/A",
                "Type": col_type,
            }
        )

    return pd.DataFrame(results)


def compute_vif(df):
    """compute variance inflation factor for numeric variables"""
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from statsmodels.tools.tools import add_constant

    X = df.select_dtypes(include=[np.number]).copy()

    if target_column in X.columns:
        X = X.drop(columns=[target_column])

    X_const = add_constant(X)

    vif_data = pd.DataFrame()
    vif_data["Variable"] = X_const.columns
    vif_data["VIF"] = [
        variance_inflation_factor(X_const.values, i)
        for i in range(X_const.shape[1])
    ]

    vif_data = vif_data[vif_data["Variable"] != "const"].reset_index(drop=True)

    return vif_data.sort_values("VIF", ascending=False)


def compute_feature_correlation_matrix(df):
    """compute pearson correlation matrix between input features"""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if target_column in numeric_cols:
        numeric_cols.remove(target_column)

    corr_matrix = df[numeric_cols].corr(method="pearson").round(4)

    return corr_matrix


def main():

    df = pd.read_excel(input_file_path)

    pre_stats = compute_descriptive_stats(df)

    df_transformed = df.copy()

    for col, func in binary_bin_cols.items():
        if col in df_transformed.columns:
            df_transformed[col] = df_transformed[col].apply(func)

    for col in new_categorical_cols:
        if col in df_transformed.columns:
            try:
                df_transformed[col] = pd.to_numeric(
                    df_transformed[col], errors="raise"
                )
            except:
                pass

    if gender_col in df_transformed.columns:

        df_transformed[gender_col] = pd.to_numeric(
            df_transformed[gender_col], errors="coerce"
        )

        df_transformed[gender_col] = df_transformed[gender_col].map(
            lambda x: gender_mapping.get(x, 2)
        )

        df_transformed.rename(
            columns={gender_col: "Gender(0=f, 1=m, 2=div)"},
            inplace=True,
        )

    for col in extreme_skew_cols:
        if col not in df_transformed.columns:
            continue

        df_transformed[col] = np.log1p(df_transformed[col])

        lower, upper = df_transformed[col].quantile([0.005, 0.995])

        df_transformed[col] = np.clip(df_transformed[col], lower, upper)

    for col in moderate_skew_cols:
        if col not in df_transformed.columns:
            continue

        df_transformed[col] = np.log1p(df_transformed[col])

        lower, upper = df_transformed[col].quantile([0.01, 0.99])

        df_transformed[col] = np.clip(df_transformed[col], lower, upper)

    for col in proportion_cols:
        if col not in df_transformed.columns:
            continue

        if col == "Percentage videos watched on day":
            df_transformed[col] = 1 - df_transformed[col]

        df_transformed[col] = np.arcsin(np.sqrt(df_transformed[col]))

    for col in df_transformed.select_dtypes(include=["object"]).columns:

        df_transformed[col] = df_transformed[col].replace(
            ["NA", "N/A", "na", "n/a"], np.nan
        )

    for col in df_transformed.columns:

        if df_transformed[col].dtype == "object":

            try:
                df_transformed[col] = pd.to_numeric(
                    df_transformed[col], errors="raise"
                )

            except:
                pass

    numeric_cols_for_impute = (
        df_transformed.select_dtypes(include=[np.number])
        .columns
        .tolist()
    )

    df_numeric = df_transformed[numeric_cols_for_impute].copy()

    imputer = KNNImputer(n_neighbors=5)

    df_imputed_numeric = pd.DataFrame(
        imputer.fit_transform(df_numeric),
        columns=df_numeric.columns,
        index=df_numeric.index,
    )

    df_transformed[numeric_cols_for_impute] = df_imputed_numeric

    favorites_vars = [
        "Avg videos not found (from favorite)",
        "Avg favorited",
    ]

    available_favorites = [
        v for v in favorites_vars if v in df_transformed.columns
    ]

    if len(available_favorites) > 0:

        X_fav = df_transformed[available_favorites].copy()

        scaler_fav = StandardScaler()

        X_fav_scaled = scaler_fav.fit_transform(X_fav)

        pca_fav = PCA(n_components=1)

        fav_pca = pca_fav.fit_transform(X_fav_scaled).flatten()

        df_transformed["Favorites_Index"] = fav_pca

    video_vars = [
        "Avg number of videos per week",
        "Avg unique per week",
        "Avg skipped per week",
        "Avg num multiple",
        "Avg liked per week",
    ]

    available_video = [
        v for v in video_vars if v in df_transformed.columns
    ]

    if len(available_video) > 0:

        X_video = df_transformed[available_video].copy()

        scaler_video = StandardScaler()

        X_video_scaled = scaler_video.fit_transform(X_video)

        pca_video = PCA(n_components=1)

        video_pca = pca_video.fit_transform(X_video_scaled).flatten()

        df_transformed["Video_Engagement_Index"] = video_pca

    all_pca_features = available_favorites + available_video

    df_transformed.drop(columns=all_pca_features, inplace=True)

    non_continuous = [
        "Parental control",
        "TT posts (self-report)",
        "TT only friends",
        "TT sexual solicitations",
        "Meeting TT strangers offline",
        "Gender(0=f, 1=m, 2=div)",
    ] + new_categorical_cols

    cont_cols_to_scale = [
        col
        for col in df_transformed.select_dtypes(include=[np.number]).columns
        if col != target_column and col not in non_continuous
    ]

    scaler = StandardScaler()

    df_transformed[cont_cols_to_scale] = scaler.fit_transform(
        df_transformed[cont_cols_to_scale]
    )

    for col in proportion_cols:

        if col in df_transformed.columns:

            df_transformed.drop(columns=[col], inplace=True)

    leakage_cols = [
        "sex abuse y/n (CTQ)",
        "maltreatment y/n (CATS)",
        "phys abuse y/n (CTQ)",
    ]

    for col in leakage_cols:

        if col in df_transformed.columns:

            df_transformed.drop(columns=[col], inplace=True)

    post_stats = compute_descriptive_stats(df_transformed)

    vif_df = compute_vif(df_transformed)

    corr_matrix = compute_feature_correlation_matrix(df_transformed)

    with pd.ExcelWriter(output_file_path, engine="openpyxl") as writer:

        pre_stats.to_excel(
            writer,
            sheet_name="Pre_Transform_Stats",
            index=False,
        )

        df_transformed.to_excel(
            writer,
            sheet_name="Transformed_Data",
            index=False,
        )

        post_stats.to_excel(
            writer,
            sheet_name="Post_Transform_Stats",
            index=False,
        )

        vif_df.to_excel(
            writer,
            sheet_name="VIF_Results",
            index=False,
        )

        corr_matrix.to_excel(
            writer,
            sheet_name="Feature_Correlation_Matrix",
        )


if __name__ == "__main__":
    main() 

In [ ]:
# step 1: imports
import os
import random
import json
import joblib
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
import numpy as np

np.random.seed(RANDOM_SEED)

# data handling and modeling
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, LeaveOneOut
from sklearn.metrics import (
    f1_score,
    make_scorer,
    confusion_matrix,
    classification_report,
    matthews_corrcoef,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# XGBoost
from xgboost import XGBClassifier

# Interpretability model
from interpret.glassbox import ExplainableBoostingClassifier

# bayesian optimization for hyperparameter tuning
import optuna
from optuna.pruners import SuccessiveHalvingPruner

# model explainability
import shap

# lightweight WGAN (PyTorch)
import torch
import torch.nn as nn
import torch.optim as optim

# save pip freeze for reproducibility
try:
    with open("pip-freeze.txt", "w") as f:
        import subprocess

        subprocess.run(["pip", "freeze"], stdout=f)
except Exception:
    pass

# plotting
import seaborn as sns
import matplotlib.pyplot as plt

# metrics
from sklearn.metrics import confusion_matrix, classification_report, matthews_corrcoef

# scorer for model evaluation
macro_f1_scorer = make_scorer(f1_score, average="macro")

# track reproducibility seeds
seeds_info = {"global_seed": RANDOM_SEED}

In [ ]:
# step 2: loading data and preparing features/target

DATA_PATH = Path("../../data/demo/dummy_transformed_dataset.xlsx")
TARGET_COLUMN = "maltreatment y/n (CTQ)"
FEATURE_COLUMNS = [
    "Avg searches",
    "Alter",
    "SES_SUM",
    "Video_Engagement_Index",
    "Favorites_Index",
    "Avg posts (count & manual) (AC)",
    "Avg session length (seconds)",
    "Parental control",
    "TT sexual solicitations",
    "Follower",
    "Gender(0=f, 1=m, 2=div)",
    "Avg no sessions",
    "Meeting TT strangers offline",
    "Being Liked (AC)",
    "TT challenges / trends",
    "Avg direct messages received",
    "Avg comments",
    "Avg single contacts",
    "Avg no blocked",
    "TT only friends",
    "Avg chat partners",
    "Private account y/n (AC)",
    "No weeks favorites recorded",
    "Following",
]

# loading workbook
if not Path(DATA_PATH).exists():
    raise FileNotFoundError(
        f"Data file not found at {DATA_PATH}. Please set DATA_PATH to the correct Excel file."
    )

df = pd.read_excel(DATA_PATH, sheet_name="Transformed_Data")
print("Data shape:", df.shape)
print(df.dtypes.value_counts())

# basic checks and feature column setup
if FEATURE_COLUMNS is None:
    if TARGET_COLUMN not in df.columns:
        raise ValueError(
            f"TARGET_COLUMN '{TARGET_COLUMN}' not found in the dataframe. Please set TARGET_COLUMN and FEATURE_COLUMNS."
        )
    FEATURE_COLUMNS = [c for c in df.columns if c != TARGET_COLUMN]

# verifying provided feature columns exist
missing_feats = [c for c in FEATURE_COLUMNS if c not in df.columns]
if missing_feats:
    raise ValueError("Feature columns missing in dataframe: " + str(missing_feats))

X_raw = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

print("Target distribution:")
print(y.value_counts())

# checking for missing values
print("\nMissing values per column:")
print(X_raw.isna().sum())

In [ ]:
# step 3: pre-processing pipeline

# identifying numerical and categorical columns automatically
numeric_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_raw.columns if c not in numeric_cols]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, cat_cols)
], remainder='drop')

# a wrapper pipeline (preprocessor only); model(s) will be appended later as needed
full_pipeline = Pipeline([('preprocessor', preprocessor)])

In [ ]:
# step 4: gan helper for tiny data (wasserstein gan - lightweight)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('using device for GAN:', DEVICE)

class Generator(nn.Module):
    def __init__(self, latent_dim, out_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, out_dim)
        )
    def forward(self, z):
        return self.net(z)

class Critic(nn.Module):
    def __init__(self, in_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x)

def fit_wgan(train_X, train_y, minority_label=1, latent_dim=8, n_epochs=200, batch_size=16,
             lr=5e-4, n_critic=5, weight_clip=0.01, device=DEVICE, patience=20, gen_hidden=64, seed=RANDOM_SEED):
    """
    Fitting a lightweight WGAN on the minority-class rows of train_X/train_y.

    Parameters:
        minority_label: label value to treat as minority class (default: 1)
        latent_dim: size of the generator's input noise vector (default: 8)
        n_epochs: number of training epochs (default: 200)
        batch_size: samples per batch (default: 16)
        lr: learning rate for both generator and critic (default: 5e-4)
        n_critic: number of critic updates per generator update (default: 5)
        weight_clip: max absolute value for critic weights (default: 0.01)
        device: torch device ('cuda' or 'cpu') (default: DEVICE)
        patience: early stopping epochs if no improvement (default: 20)
        gen_hidden: hidden layer size for generator and critic (default: 64)
        seed: random seed for reproducibility (default: RANDOM_SEED)
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    if isinstance(train_X, pd.DataFrame):
        cols = train_X.columns
        X_vals = train_X.values.astype(np.float32)
    else:
        cols = None
        X_vals = np.asarray(train_X, dtype=np.float32)

    y_vals = np.asarray(train_y)
    minority_idx = np.where(y_vals == minority_label)[0]
    if len(minority_idx) == 0:
        # nothing to do
        def null_generator(n_syn):
            return np.empty((0, X_vals.shape[1])), np.empty((0,))
        return null_generator

    X_min = X_vals[minority_idx]
    n_min = X_min.shape[0]

    in_dim = X_min.shape[1]
    G = Generator(latent_dim=latent_dim, out_dim=in_dim, hidden_dim=gen_hidden).to(device)
    D = Critic(in_dim=in_dim, hidden_dim=gen_hidden).to(device)

    opt_G = optim.RMSprop(G.parameters(), lr=lr)
    opt_D = optim.RMSprop(D.parameters(), lr=lr)

    dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_min))
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    best_critic_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(n_epochs):
        D_losses = []
        for i_batch, (real_batch,) in enumerate(loader):
            real_batch = real_batch.to(device)
            # training critic n_critic times
            for _ in range(n_critic):
                z = torch.randn((real_batch.size(0), latent_dim), device=device)
                fake = G(z).detach()
                D_real = D(real_batch).mean()
                D_fake = D(fake).mean()
                loss_D = -(D_real - D_fake)
                opt_D.zero_grad()
                loss_D.backward()
                opt_D.step()
                # weight clipping
                for p in D.parameters():
                    p.data.clamp_(-weight_clip, weight_clip)
                D_losses.append(loss_D.item())

            # training generator
            z = torch.randn((real_batch.size(0), latent_dim), device=device)
            fake = G(z)
            loss_G = -D(fake).mean()
            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

        mean_D_loss = float(np.mean(D_losses)) if D_losses else 0.0
        # early-stopping based on critic loss
        if mean_D_loss < best_critic_loss - 1e-6:
            best_critic_loss = mean_D_loss
            epochs_no_improve = 0
            # saving snapshot
            best_state = {
                'G': G.state_dict(),
                'D': D.state_dict()
            }
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            break

    # loading best generator
    try:
        G.load_state_dict(best_state['G'])
    except Exception:
        pass

    G.eval()

    def gen(n_syn):
        if n_syn <= 0:
            return (pd.DataFrame(columns=cols) if cols is not None else np.empty((0, in_dim))), np.empty((0,), dtype=int)
        z = torch.randn((n_syn, latent_dim), device=device)
        with torch.no_grad():
            out = G(z).cpu().numpy()
        if cols is not None:
            return pd.DataFrame(out, columns=cols), np.ones((out.shape[0],), dtype=int)
        else:
            return out, np.ones((out.shape[0],), dtype=int)

    return gen

In [ ]:
# step 5: Coarse CV search with all possible candidates

# 5.1 defining candidate learners and search spaces

def model_and_space_factory(seed=RANDOM_SEED):
    spaces, factories = {}, {}

    # logistic regression
    def make_logreg(params):
        return LogisticRegression(
            penalty='l2', solver='lbfgs', max_iter=1000,
            C=params.get('C', 1.0),
            class_weight=params.get('class_weight', None),
            random_state=seed
        )

    def space_logreg(trial):
        return {
            'C': trial.suggest_float('C', 1e-4, 100.0, log=True),
            'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced'])
        }

    factories['LogisticRegression'] = make_logreg
    spaces['LogisticRegression'] = space_logreg

    # random forest
    def make_rf(params):
        return RandomForestClassifier(
            n_estimators=int(params.get('n_estimators', 100)),
            max_depth=None if params.get('max_depth') is None else int(params.get('max_depth')),
            min_samples_leaf=int(params.get('min_samples_leaf', 1)),
            class_weight=params.get('class_weight', None),
            random_state=seed
        )

    def space_rf(trial):
        return {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth': trial.suggest_int('max_depth', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced'])
        }

    factories['RandomForest'] = make_rf
    spaces['RandomForest'] = space_rf

    # XGBoost
    def make_xgb(params):
        if XGBClassifier is None:
            raise ImportError('XGBoost not installed')
        return XGBClassifier(
            learning_rate=params.get('learning_rate', 0.1),
            n_estimators=int(params.get('n_estimators', 100)),
            max_depth=int(params.get('max_depth', 3)),
            subsample=params.get('subsample', 1.0),
            colsample_bytree=params.get('colsample_bytree', 1.0),
            reg_lambda=params.get('reg_lambda', 1.0),
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=seed
        )

    def space_xgb(trial):
        return {
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.5, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth': trial.suggest_int('max_depth', 2, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True)
        }

    factories['XGBoost'] = make_xgb
    spaces['XGBoost'] = space_xgb

    # SVC
    def make_svc(params):
        return SVC(
            C=params.get('C', 1.0),
            gamma=params.get('gamma', 'scale'),
            class_weight=params.get('class_weight', None),
            probability=True,
            random_state=seed
        )

    def space_svc(trial):
        return {
            'C': trial.suggest_float('C', 1e-4, 100.0, log=True),
            'gamma': trial.suggest_categorical('gamma', ['scale', 'auto']),
            'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced'])
        }

    factories['SVC'] = make_svc
    spaces['SVC'] = space_svc

    # ExtraTrees
    def make_et(params):
        return ExtraTreesClassifier(
            n_estimators=int(params.get('n_estimators', 100)),
            max_features=params.get('max_features', 'sqrt'),
            min_samples_split=int(params.get('min_samples_split', 2)),
            random_state=seed
        )

    def space_et(trial):
        return {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10)
        }

    factories['ExtraTrees'] = make_et
    spaces['ExtraTrees'] = space_et

    # Conditional Inference Tree
    def make_cit(params):
        return DecisionTreeClassifier(
            max_depth=None if params.get('max_depth') is None else int(params.get('max_depth')),
            min_samples_leaf=int(params.get('min_samples_leaf', 1)),
            random_state=seed
        )

    def space_cit(trial):
        return {
            'max_depth': trial.suggest_int('max_depth', 1, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10)
        }

    factories['ConditionalInferenceTree'] = make_cit
    spaces['ConditionalInferenceTree'] = space_cit

    # EBM
    if ExplainableBoostingClassifier is not None:
        def make_ebm(params):
            return ExplainableBoostingClassifier(
                max_bins=int(params.get('max_bins', 255)),
                learning_rate=params.get('learning_rate', 0.01),
                random_state=seed
            )

        def space_ebm(trial):
            return {
                'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.5, log=True),
                'n_estimators': trial.suggest_int('n_estimators', 10, 500),
                'max_bins': trial.suggest_int('max_bins', 32, 255)
            }

        factories['EBM'] = make_ebm
        spaces['EBM'] = space_ebm

    return factories, spaces


model_factories, optuna_spaces = model_and_space_factory()


# Step 5.2: Outer stratified Leave-One-Out loop

n_samples = X_raw.shape[0]
results_outer = []

X_df = X_raw.reset_index(drop=True)
y_series = y.reset_index(drop=True)

for test_idx in range(n_samples):

    train_idx = [i for i in range(n_samples) if i != test_idx]

    X_train_raw = X_df.iloc[train_idx].reset_index(drop=True)
    y_train = y_series.iloc[train_idx].reset_index(drop=True)

    X_test_raw = X_df.iloc[[test_idx]].reset_index(drop=True)
    y_test = y_series.iloc[[test_idx]].reset_index(drop=True)

    pipeline = Pipeline([('preprocessor', preprocessor)])

    X_train = pipeline.fit_transform(X_train_raw)

    try:
        ohe = pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
        cat_feature_names = list(ohe.get_feature_names_out(cat_cols)) if len(cat_cols) > 0 else []
    except Exception:
        cat_feature_names = []

    transformed_cols = numeric_cols + cat_feature_names

    X_train_df = pd.DataFrame(X_train, columns=transformed_cols)

    X_test = pipeline.transform(X_test_raw)
    X_test_df = pd.DataFrame(X_test, columns=transformed_cols)

    gen = fit_wgan(
        X_train_df, y_train,
        minority_label=1,
        latent_dim=8,
        n_epochs=200,
        batch_size=8,
        lr=5e-4,
        n_critic=3,
        weight_clip=0.01,
        device=DEVICE,
        patience=30,
        seed=RANDOM_SEED
    )

    n_minority = int((y_train == 1).sum())
    N_syn = max(1, n_minority)

    X_syn_df, y_syn = gen(N_syn)

    if isinstance(X_syn_df, pd.DataFrame):
        X_aug = pd.concat([X_train_df, X_syn_df], ignore_index=True)
    else:
        X_aug = pd.DataFrame(X_train_df.values, columns=transformed_cols)

    y_aug = pd.concat([y_train.reset_index(drop=True), pd.Series(y_syn)], ignore_index=True)

    X_arr = X_aug.values if isinstance(X_aug, pd.DataFrame) else np.asarray(X_aug)
    y_arr = np.asarray(y_aug)

    for model_name in model_factories:

        if model_name == 'XGBoost' and XGBClassifier is None:
            continue
        if model_name == 'EBM' and ExplainableBoostingClassifier is None:
            continue

        def make_objective(model_name):

            def objective(trial):

                params = optuna_spaces[model_name](trial)
                model = model_factories[model_name](params)

                skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

                scores = []

                for train_i, val_i in skf.split(X_arr, y_arr):

                    model.fit(X_arr[train_i], y_arr[train_i])
                    y_pred = model.predict(X_arr[val_i])

                    scores.append(f1_score(y_arr[val_i], y_pred, average='macro'))

                return float(np.mean(scores))

            return objective

        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED)
        )

        try:
            study.optimize(make_objective(model_name), n_trials=5, n_jobs=1) # was originally 20
        except Exception:
            pass

        best_params = study.best_params if study.best_trial is not None else {}

        model_best = model_factories[model_name](best_params)
        model_best.fit(X_arr, y_arr)

        y_pred_i = model_best.predict(X_test_df.values)

        outer_f1 = f1_score(y_test, y_pred_i, average='macro')

        results_outer.append({
            'test_idx': test_idx,
            'model_name': model_name,
            'best_params': best_params,
            'outer_macro_f1': float(outer_f1)
        })


# Step 5.3: aggregating outer-fold results

results_df = pd.DataFrame(results_outer)

summary = {}

for model_name, group in results_df.groupby('model_name'):

    scores = group['outer_macro_f1'].values
    mean_score = float(np.mean(scores))

    rng = np.random.RandomState(RANDOM_SEED)

    boots = [rng.choice(scores, size=len(scores), replace=True).mean() for _ in range(1000)]

    lo, hi = np.percentile(boots, [2.5, 97.5])

    summary[model_name] = {
        'mean_macro_f1': mean_score,
        'ci_95': (float(lo), float(hi)),
        'n_folds': len(scores)
    }

summary_df = pd.DataFrame(summary).T
summary_df = summary_df.sort_values('mean_macro_f1', ascending=False)

print(summary_df)

best_model_name = summary_df.index[0]
runner_up_name = summary_df.index[1] if summary_df.shape[0] > 1 else None

print('Best model selected:', best_model_name)

In [ ]:
# step 5.4: refining XGBoost parameters & optuna search spaces

def model_and_space_factory(seed=RANDOM_SEED):
    spaces = {}
    factories = {}

    # xgboost (active - refined search space)
    def make_xgb(params):
        if XGBClassifier is None:
            raise ImportError("XGBoost not installed")
        return XGBClassifier(
            learning_rate=params.get(
                "learning_rate", 0.003
            ),
            n_estimators=int(
                params.get("n_estimators", 350)
            ),
            max_depth=int(
                params.get("max_depth", 4)
            ),
            subsample=params.get(
                "subsample", 0.8
            ),
            colsample_bytree=params.get(
                "colsample_bytree", 0.65
            ),
            reg_lambda=params.get(
                "reg_lambda", 0.05
            ),
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=seed,
        )

    def space_xgb(trial):
        lr = trial.suggest_float(
            "learning_rate", 0.0005, 0.01, log=True
        )
        n_estimators = trial.suggest_int("n_estimators", 200, 600)
        max_depth = trial.suggest_int("max_depth", 3, 6)
        subsample = trial.suggest_float("subsample", 0.7, 0.9)
        colsample_bytree = trial.suggest_float(
            "colsample_bytree", 0.5, 0.8
        )
        reg_lambda = trial.suggest_float(
            "reg_lambda", 0.001, 1.0, log=True
        )
        return {
            "learning_rate": lr,
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "reg_lambda": reg_lambda,
        }

    factories["XGBoost"] = make_xgb
    spaces["XGBoost"] = space_xgb

    return factories, spaces

model_factories, optuna_spaces = model_and_space_factory()

In [ ]:
# step 6: outer stratified leave-one-out loop (model-type selection)

n_samples = X_raw.shape[0]
results_outer = []  # list of dicts: {model_name, best_params, outer_macro_f1, test_idx}

# converting X_raw to DataFrame to retain columns
X_df = X_raw.reset_index(drop=True)
y_series = y.reset_index(drop=True)

for test_idx in range(n_samples):
    # a) creating training and test
    train_idx = [i for i in range(n_samples) if i != test_idx]
    X_train_raw = X_df.iloc[train_idx].reset_index(drop=True)
    y_train = y_series.iloc[train_idx].reset_index(drop=True)
    X_test_raw = X_df.iloc[[test_idx]].reset_index(drop=True)
    y_test = y_series.iloc[[test_idx]].reset_index(drop=True)

    # b) fitting preprocessing on X_train_raw
    pipeline = Pipeline([("preprocessor", preprocessor)])
    X_train = pipeline.fit_transform(X_train_raw)
    # transforming returns numpy array; convert to DataFrame for GAN column names
    try:
        # get feature names from ColumnTransformer
        ohe = (
            pipeline.named_steps["preprocessor"]
            .named_transformers_["cat"]
            .named_steps["onehot"]
        )
        cat_feature_names = []
        if len(cat_cols) > 0:
            cat_feature_names = list(ohe.get_feature_names_out(cat_cols))
    except Exception:
        cat_feature_names = []

    transformed_cols = numeric_cols + cat_feature_names
    X_train_df = pd.DataFrame(X_train, columns=transformed_cols)
    X_test = pipeline.transform(X_test_raw)
    X_test_df = pd.DataFrame(X_test, columns=transformed_cols)

    # c) fitting GAN on minority class of transformed training data
    gen = fit_wgan(
        X_train_df,
        y_train,
        minority_label=1,
        latent_dim=8,
        n_epochs=200,
        batch_size=8,
        lr=5e-4,
        n_critic=3,
        weight_clip=0.01,
        device=DEVICE,
        patience=30,
        seed=RANDOM_SEED,
    )
    # generating roughly same number as minority present
    n_minority = int((y_train == 1).sum())
    N_syn = max(1, n_minority)
    X_syn_df, y_syn = gen(N_syn)

    # d) augmenting training data
    if isinstance(X_syn_df, pd.DataFrame):
        X_aug = pd.concat([X_train_df, X_syn_df], ignore_index=True)
    else:
        X_aug = pd.DataFrame(X_train_df.values, columns=transformed_cols)
    y_aug = pd.concat(
        [y_train.reset_index(drop=True), pd.Series(y_syn)], ignore_index=True
    )

    # e) for each candidate learner — only XGBoost now
    for model_name, factory in model_factories.items():
        if model_name == "XGBoost" and XGBClassifier is None:
            continue

        # inner CV objective
        def make_objective(model_name, X_aug, y_aug, pipeline_cols, seed=RANDOM_SEED):
            def objective(trial):
                params = optuna_spaces[model_name](trial)
                model = model_factories[model_name](params)
                # building full pipeline: the preprocessing is already applied; we'll fit model directly on X_aug numpy
                skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
                scores = []
                X_arr = (
                    X_aug.values
                    if isinstance(X_aug, pd.DataFrame)
                    else np.asarray(X_aug)
                )
                y_arr = np.asarray(y_aug)
                for train_i, val_i in skf.split(X_arr, y_arr):
                    X_tr, X_val = X_arr[train_i], X_arr[val_i]
                    y_tr, y_val = y_arr[train_i], y_arr[val_i]
                    model.fit(X_tr, y_tr)
                    y_pred = model.predict(X_val)
                    scores.append(f1_score(y_val, y_pred, average="macro"))
                return float(np.mean(scores))

            return objective

        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(
                seed=RANDOM_SEED
            ),
            pruner=SuccessiveHalvingPruner()
        )
        try:
            study.optimize(
                make_objective(model_name, X_aug, y_aug, transformed_cols),
                n_trials=5, # was originally 20
                n_jobs=1,
            )
        except Exception as e:
            # allback to default params in case of failures during optimization
            print(f"optimization failed for {model_name}: {e}")
            study = study  # keeping existing (will use defaults)

        best_params = study.best_params if study.best_trial is not None else {}
        # refitting model on full augmented training set
        model_best = model_factories[model_name](best_params)
        model_best.fit(
            X_aug.values if isinstance(X_aug, pd.DataFrame) else X_aug,
            np.asarray(y_aug),
        )
        y_pred_i = model_best.predict(X_test_df.values)
        outer_f1 = f1_score(y_test, y_pred_i, average="macro")

        results_outer.append(
            {
                "test_idx": test_idx,
                "model_name": model_name,
                "best_params": best_params,
                "outer_macro_f1": float(outer_f1),
            }
        )

In [ ]:
# step 7: aggregating outer-fold results to summarize model performance across all leave-one-out folds.

results_df = pd.DataFrame(results_outer)
summary = {}
for model_name, group in results_df.groupby("model_name"):
    scores = group["outer_macro_f1"].values
    mean_score = float(np.mean(scores))
    # bootstrap 95% CI
    rng = np.random.RandomState(RANDOM_SEED)
    boots = []
    for _ in range(1000):
        samp = rng.choice(scores, size=len(scores), replace=True)
        boots.append(samp.mean())
    lo, hi = np.percentile(boots, [2.5, 97.5])
    summary[model_name] = {
        "mean_macro_f1": mean_score,
        "ci_95": (float(lo), float(hi)),
        "n_folds": len(scores),
    }

summary_df = pd.DataFrame({k: v for k, v in summary.items()}).T
summary_df = summary_df.sort_values("mean_macro_f1", ascending=False)
print(summary_df)

best_model_name = summary_df.index[0]
runner_up_name = summary_df.index[1] if summary_df.shape[0] > 1 else None
print("best model selected:", best_model_name)

output_file = Path("results/LOOCV_XGBoost_dummy_evaluation_results.xlsx")
output_file.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_file) as writer:
    # save outer loop results
    results_df.to_excel(writer, sheet_name="Outer_Fold_Results", index=False)

    # save summary statistics
    summary_df.to_excel(writer, sheet_name="Summary_Stats")

print(f"Results saved to {output_file}")

best_model_name = summary_df.index[0]
best_model_results = [r for r in results_outer if r['model_name'] == best_model_name]

In [ ]:
# step 8: SHAP

# refined feature name mapping
feature_name_mapping = {
    "Avg searches": "Searches/Week",
    "Alter": "Age",
    "SES_SUM": "Socioeconomic Status",
    "Avg session length (seconds)": "Session Length (Seconds)",
    "Avg posts (count & manual) (AC)": "Posts/Week",
    "Parental control": "Parental Control",
    "TT sexual solicitations": "Sexual Solicitations On TikTok",
    "No weeks favorites recorded": "Number Of Weeks Favorites Recorded",
    "Follower": "Follower Count",
    "Avg chat partners": "Chat Partners/Week",
    "Avg direct messages received": "Direct Messages Received/Week",
    "Meeting TT strangers offline": "Meetings TikTok-Strangers Offline",
    "Avg single contacts": "Single Contacts/Week",
    "Favorites_Index": "Favorites Index",
    "Following": "Following Count",
    "Avg no blocked": "Accounts Blocked/Week",
    "Being Liked (AC)": "Likes Received on Profile",
    "TT challenges / trends": "Participation in TikTok Trend/Challenges",
    "Avg no sessions": "Sessions/Week",
    "Avg comments": "Comments/Week",
    "Private account y/n (AC)": "Private Account Status",
    "TT only friends": "Online-Only Friends",
    "Video_Engagement_Index": "Video Engagement Index",
}

# collecting all test data for SHAP
X_test_concat = []  # to store test data for SHAP

for result in results_outer:
    test_idx = result['test_idx']

    # splitting data for this fold
    train_idx = [i for i in range(n_samples) if i != test_idx]
    X_train_raw = X_df.iloc[train_idx].reset_index(drop=True)
    y_train = y_series.iloc[train_idx].reset_index(drop=True)
    X_test_raw = X_df.iloc[[test_idx]].reset_index(drop=True)

    # applying preprocessing
    pipeline = Pipeline([("preprocessor", preprocessor)])
    X_train = pipeline.fit_transform(X_train_raw)
    X_test = pipeline.transform(X_test_raw)

    # collecting test data
    X_test_df = pd.DataFrame(X_test, columns=transformed_cols)
    X_test_concat.append(X_test_df)

# concatenating all test data
X_test_concat = pd.concat(X_test_concat, ignore_index=True)

# fitting preprocessing on the full dataset
full_pipeline = Pipeline([("preprocessor", preprocessor)])
X_all_trans = full_pipeline.fit_transform(X_df)

try:
    ohe = (
        full_pipeline.named_steps["preprocessor"]
        .named_transformers_["cat"]
        .named_steps["onehot"]
    )
    cat_feature_names = (
        list(ohe.get_feature_names_out(cat_cols)) if len(cat_cols) > 0 else []
    )
except Exception:
    cat_feature_names = []
transformed_cols = numeric_cols + cat_feature_names
X_all_df = pd.DataFrame(X_all_trans, columns=transformed_cols)

# fitting GAN on full training (minority) and augmenting
gen_full = fit_wgan(
    X_all_df,
    y_series,
    minority_label=1,
    latent_dim=8,
    n_epochs=200,
    batch_size=8,
    lr=5e-4,
    n_critic=3,
    weight_clip=0.01,
    device=DEVICE,
    patience=30,
    seed=RANDOM_SEED,
)
n_minority_full = int((y_series == 1).sum())
X_syn_full, y_syn_full = gen_full(n_minority_full)
X_aug_full = pd.concat([X_all_df, X_syn_full], ignore_index=True)
y_aug_full = pd.concat(
    [y_series.reset_index(drop=True), pd.Series(y_syn_full)], ignore_index=True
)

# training final model (using best parameters from the first fold)
final_params = best_model_results[0]["best_params"]
final_model = model_factories[best_model_name](final_params)
final_model.fit(X_aug_full, np.asarray(y_aug_full))

# defining prediction function for SHAP
def predict_proba_class1(X):
    if isinstance(X, np.ndarray):
        X_df = pd.DataFrame(X, columns=transformed_cols)
    elif isinstance(X, pd.DataFrame):
        X_df = X.copy()
    else:
        X_df = pd.DataFrame(X, columns=transformed_cols)
    probs = final_model.predict_proba(X_df)
    if probs.ndim == 2 and probs.shape[1] > 1:
        return probs[:, 1]
    else:
        return probs.ravel()

# preparing background data for SHAP
background_df = shap.sample(X_aug_full, nsamples=50, random_state=RANDOM_SEED)
if not isinstance(background_df, pd.DataFrame):
    background_df = pd.DataFrame(background_df, columns=transformed_cols)

# computing SHAP values
shap_vals_arr = None
explainer_used = None
try:
    explainer = shap.Explainer(predict_proba_class1, background_df)
    shap_exp = explainer(X_test_concat)
    shap_vals_arr = shap_exp.values
    explainer_used = "shap.Explainer"
except Exception as e:
    try:
        kernel_expl = shap.KernelExplainer(predict_proba_class1, background_df)
        shap_vals = kernel_expl.shap_values(X_test_concat, nsamples=100)
        if isinstance(shap_vals, list):
            shap_vals_arr = np.array(shap_vals[0])
        else:
            shap_vals_arr = np.array(shap_vals)
        explainer_used = "shap.KernelExplainer"
    except Exception as e2:
        print("both shap.Explainer and KernelExplainer failed:", e, e2)

# plotting SHAP summary
if shap_vals_arr is not None:
    shap_summary = pd.DataFrame(
        {
            "feature": transformed_cols,
            "mean_abs_shap": np.mean(np.abs(shap_vals_arr), axis=0),
        }
    )
    shap_summary = shap_summary.sort_values("mean_abs_shap", ascending=False)
    print(f"using explainer: {explainer_used}")
    print("\nTop features by mean absolute SHAP:")

    # mapping feature name to the summary data
    shap_summary_mapped = shap_summary.copy()
    shap_summary_mapped["feature"] = (
        shap_summary_mapped["feature"]
        .map(feature_name_mapping)
        .fillna(shap_summary_mapped["feature"])
    )
    print(shap_summary_mapped.head(10))

    # mapping the actual column names that will be used in plots
    plot_feature_names = []
    for col in transformed_cols:
        if col in feature_name_mapping:
            plot_feature_names.append(feature_name_mapping[col])
        else:
            plot_feature_names.append(col)

    try:
        shap.summary_plot(
            shap_vals_arr,
            X_test_concat,
            feature_names=plot_feature_names,
            show=False,
        )
    except Exception as e:
        print("could not produce SHAP summary plot:", e)
else:
    print("SHAP values not computed; skipping SHAP summary.")

# bar plot of mean absolute SHAP values
if shap_vals_arr is not None:
    plt.figure(figsize=(10, 6))

    # mapping feature names for the bar plot
    shap_summary_mapped = shap_summary.copy()
    shap_summary_mapped["feature"] = (
        shap_summary_mapped["feature"]
        .map(feature_name_mapping)
        .fillna(shap_summary_mapped["feature"])
    )

    # flipping to descending order
    shap_summary_sorted = shap_summary_mapped.sort_values(
        "mean_abs_shap", ascending=False
    )

    sns.barplot(
        x="mean_abs_shap",
        y="feature",
        data=shap_summary_sorted,
        palette="viridis",
    )
    plt.title("Mean Absolute SHAP Values by Feature")
    plt.xlabel("Mean Absolute SHAP Value")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print("SHAP values not computed; skipping bar plot.")